In [1]:
from processing.auto_cluster_dl import auto_cluster_dl_hdbscan
from processing.clahe import chalhe_images
from processing.make_crops import make_crops
from processing.megadetector_step import megadetector_classify
from processing.divide import divide_images
from processing.remove_footer import remove_footer
import os
import shutil
import numpy as np

# --- CONFIGURACIÓN ---
PATH = os.getcwd()
SRC_IMAGES = os.path.join(PATH, 'dataset_ecuador')
IMAGES = os.path.join(PATH, 'images')
# Carpetas intermedias
SORTED_DIR = os.path.join(PATH, 'images_sorted')
CROPS_RAW_DIR = os.path.join(PATH, 'crops')
CROPS_CLAHE_DIR = os.path.join(PATH, 'crops_clahe_processed')
CLUSTERS_OUTPUT = os.path.join(PATH, 'Clusters')       # Salida final
# Definimos rutas claras

ANIMALS_FOLDER = 'Animales'
EMPTY_FOLDER = 'Vacias'

path_animales_img = os.path.join(SORTED_DIR, ANIMALS_FOLDER)

/home/ariel/Desktop/iwildcam_preprocessing_pipeline/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Pipeline

In [ ]:
# --- PIPELINE ---

print("=== INICIANDO PIPELINE (MegaDetector -> Crop -> CLAHE -> ResNet50) ===")

# 0. RECORTAR FOOTER
remove_footer(
    input_folder=SRC_IMAGES,
    output_folder=IMAGES,
    pixels_to_cut=400,
    quality=95
)

In [ ]:
# 1. MEGADETECTOR
print("\n--- Paso 1: MegaDetector ---")
if os.path.exists('resultados_megadetector.json'):
    print("JSON detectado. Saltando.")
else:
    megadetector_classify(
        input_folder=IMAGES,
        output_file='resultados_megadetector.json',
        model_version='MDV5A',
        conf_threshold=0.2,
        recursive=True
    )

In [ ]:
# 2. DIVIDE
print("\n--- Paso 2: Dividir ---")
divide_images(
    json_file='resultados_megadetector.json',
    source_folder=IMAGES,
    dest_root=SORTED_DIR,
    animals_folder_name=ANIMALS_FOLDER,
    empty_folder_name=EMPTY_FOLDER,
    conf_threshold=0.4,
    accepted_categories=['1']
)

shutil.rmtree(IMAGES)

In [ ]:
# 3. MAKE CROPS
print("\n--- Paso 3: Recortes (Crops) ---")

make_crops(
    json_file='resultados_megadetector.json',
    input_folder=path_animales_img,
    output_folder=CROPS_RAW_DIR,
    conf_threshold=0.4,
    accepted_categories=['1']
)

shutil.rmtree(SORTED_DIR)

In [ ]:
# 4. CLAHE
print("\n--- Paso 4: CLAHE ---")

chalhe_images(
    input_dir=CROPS_RAW_DIR,
    output_dir=CROPS_CLAHE_DIR
)

shutil.rmtree(CROPS_RAW_DIR)

In [2]:
# 5. AUTO CLUSTER (RESNET)
print("\n--- Paso 5: Clustering con ResNet50 + UMAP ---")
# CAMBIO: Usamos la función que integra ResNet
clusterer, embedding = auto_cluster_dl_hdbscan(
    input_dir=CROPS_CLAHE_DIR,    # Usamos los crops mejorados
    output_dir=CLUSTERS_OUTPUT,
    min_cluster_size=15
)

print(f"\n¡LISTO! Revisa la carpeta: {CLUSTERS_OUTPUT}")


--- Paso 5: Clustering con ResNet50 + UMAP ---
--- Paso 1: Extracción de Features con ResNet50 ---
Features ya extraídos previamente. Cargando...
Features cargados: (1653, 1024)
--- Paso 2: Reducción de Dimensionalidad (UMAP) ---


/home/ariel/Desktop/iwildcam_preprocessing_pipeline/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/ariel/Desktop/iwildcam_preprocessing_pipeline/.venv/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


Dimensiones reducidas con UMAP: (1653, 15)
--- Paso 3: Clustering con HDBSCAN (min_cluster_size=15) ---

>>> RESULTADO: Se encontraron 31 clusters y 573 imágenes de Ruido (-1).
--- Paso 4: Organizando archivos ---


Moviendo archivos: 100%|██████████| 1653/1653 [00:00<00:00, 13242.54it/s]


--- RESUMEN FINAL ---
Cluster_00: 24 imágenes
Cluster_01: 59 imágenes
Cluster_02: 30 imágenes
Cluster_03: 24 imágenes
Cluster_04: 22 imágenes
Cluster_05: 17 imágenes
Cluster_06: 30 imágenes
Cluster_07: 15 imágenes
Cluster_08: 21 imágenes
Cluster_09: 16 imágenes
Cluster_10: 18 imágenes
Cluster_11: 38 imágenes
Cluster_12: 17 imágenes
Cluster_13: 37 imágenes
Cluster_14: 19 imágenes
Cluster_15: 33 imágenes
Cluster_16: 31 imágenes
Cluster_17: 20 imágenes
Cluster_18: 35 imágenes
Cluster_19: 58 imágenes
Cluster_20: 87 imágenes
Cluster_21: 21 imágenes
Cluster_22: 18 imágenes
Cluster_23: 34 imágenes
Cluster_24: 24 imágenes
Cluster_25: 15 imágenes
Cluster_26: 106 imágenes
Cluster_27: 55 imágenes
Cluster_28: 44 imágenes
Cluster_29: 75 imágenes
Cluster_30: 37 imágenes
Cluster_Ruido_Outliers: 573 imágenes

¡LISTO! Revisa la carpeta: /home/ariel/Desktop/iwildcam_preprocessing_pipeline/Clusters


In [3]:
from sklearn.metrics import silhouette_score

labels = clusterer.fit_predict(embedding)

# --- 3.1 MÉTRICA: SILHOUETTE (ignorando ruido) ---
mask = labels != -1

if np.sum(mask) > 1 and len(set(labels[mask])) > 1:
    silhouette = silhouette_score(
        embedding[mask],
        labels[mask],
        metric='euclidean'  # coherente con HDBSCAN en UMAP
    )
    print(f"Silhouette Score (sin ruido): {silhouette:.4f}")
else:
    print("No se puede calcular Silhouette (muy pocos clusters o puntos).")

Silhouette Score (sin ruido): 0.6737
